In [28]:
!pip install -U peft datasets sacrebleu sentencepiece accelerate evaluate bitsandbytes transformers==4.44.2 huggingface_hub==0.23.5

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from datasets import load_dataset
import random

ds = load_dataset("nhuvo/MedEV")
dataset = ds["train"]

offset = 340897
num_pairs = 340897

pairs = []

for i in range(num_pairs):
    en = dataset[i]["text"]
    vi = dataset[i + offset]["text"]

    pairs.append({
        "en": en,
        "vi": vi
    })


random.seed(42)
random.shuffle(pairs)
pairs_30k = pairs[:30000]

train_size = int(0.8 * len(pairs_30k))
val_size = int(0.1 * len(pairs_30k))

train_pairs = pairs_30k[:train_size]
val_pairs = pairs_30k[train_size:train_size+val_size]
test_pairs = pairs_30k[train_size+val_size:]

In [5]:
#test
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate

# load EnviT5
model_name = "VietAI/envit5-translation"
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

device = "cuda" if torch.cuda.is_available() else "cpu"

text = "translate English to Vietnamese: The patient has severe pneumonia."

inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


vi: Bệnh nhân bị viêm phổi nặng.


In [31]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import torch

# load EnviT5
model_name = "VietAI/envit5-translation"
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

device = "cuda" if torch.cuda.is_available() else "cpu"

# BLEU metric
bleu = evaluate.load("bleu")

predictions = []
references = []

for sample in test_pairs:
    
    src = sample["en"]      # câu tiếng Anh
    tgt = sample["vi"]      # câu tiếng Việt chuẩn
    
    inputs = tokenizer(src, return_tensors="pt", truncation=True).to(device)
    
    output_ids = model.generate(**inputs, max_new_tokens=128)
    
    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    predictions.append(pred)
    references.append([tgt])   # BLEU cần list of list

# compute BLEU
result = bleu.compute(predictions=predictions, references=references)

print("BLEU score:", result["bleu"])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


BLEU score: 0.40916875536812247
